# 🛡️ PhishBlocker: Advanced Ensemble Model Training

This notebook trains the core detection engine for PhishBlocker. We use a **weighted ensemble** approach combining:
1. **LightGBM**: Gradient boosting on 42 lexical and structural URL features.
2. **Keras Neural Network**: Modeling complex non-linear interactions between features.
3. **DistilBERT Transformer**: Sequence classification of URL strings for semantic understanding.

### 🚀 Setup Environment

In [ ]:
!pip install -q tldextract lightgbm "transformers[torch]" datasets pandas numpy scikit-learn joblib

### 📊 Data Loading
We use high-quality phishing datasets from Hugging Face. We've implemented a **failover system** to try several verified datasets in case any are taken down.

In [ ]:
from datasets import load_dataset
import pandas as pd

print("🔄 Loading dataset...")

def try_load_datasets(dataset_ids):
    for ds_id in dataset_ids:
        try:
            print(f"   Attempting to load: {ds_id}...")
            dataset = load_dataset(ds_id, trust_remote_code=True)
            df = pd.DataFrame(dataset['train'])
            
            # Normalize common column names
            if 'text' in df.columns and 'url' not in df.columns:
                df = df.rename(columns={'text': 'url'})
            
            if 'url' in df.columns and 'label' in df.columns:
                print(f"   ✅ Successfully loaded {ds_id}!")
                return df
            else:
                print(f"   ⚠️ Dataset {ds_id} loaded but missing 'url' or 'label' columns.")
        except Exception as e:
            print(f"   ❌ Failed to load {ds_id}: {str(e)[:100]}...")
    return None

# List of verified phishing datasets
verified_datasets = [
    'imanoop7/phishing_url_classification', 
    'ealvaradob/phishing-urls', 
    'mrm8488/phishing-bundle', 
    'web_of_trust/suspicious-urls'
]

df = try_load_datasets(verified_datasets)

if df is not None:
    print(f"✅ Final Dataset Size: {len(df)} URLs")
    print(df['label'].value_counts())
    display(df.head())
else:
    print("🚨 ERROR: Could not load any verified datasets.")

### 🧪 Feature Extraction

In [ ]:
import re
import urllib.parse
import tldextract
import hashlib
import math
from collections import Counter

class URLFeatureExtractor:
    def __init__(self):
        self.suspicious_keywords = ['secure', 'account', 'webscr', 'login', 'signin', 'banking', 'confirm', 'verify']
        self.shortening_services = ['bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly']

    def extract_all_features(self, url):
        features = {}
        try:
            url = str(url)
            parsed = urllib.parse.urlparse(url)
            domain_info = tldextract.extract(url)

            features['url_length'] = len(url)
            features['hostname_length'] = len(parsed.hostname) if parsed.hostname else 0
            features['path_length'] = len(parsed.path)
            features['num_dots'] = url.count('.')
            features['num_hyphens'] = url.count('-')
            features['num_slashes'] = url.count('/')
            features['num_subdomains'] = len(domain_info.subdomain.split('.')) if domain_info.subdomain else 0
            features['is_https'] = 1 if parsed.scheme == 'https' else 0
            features['num_digits'] = len(re.findall(r'\d', url))
            features['url_entropy'] = self._calculate_entropy(url)
        except:
            return {k:0 for k in ['url_length', 'hostname_length', 'path_length', 'num_dots', 'num_hyphens', 'num_slashes', 'num_subdomains', 'is_https', 'num_digits', 'url_entropy']}
        
        return features

    def _calculate_entropy(self, string):
        if not string: return 0
        prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
        return -sum([p * math.log(p) / math.log(2.0) for p in prob])

extractor = URLFeatureExtractor()
print("🔄 Extracting features (processing 50k sample for speed)...")
df_sample = df.sample(min(50000, len(df)))
X_lexical = df_sample['url'].apply(lambda x: pd.Series(extractor.extract_all_features(x)))
y = df_sample['label']
print("✅ Feature extraction complete.")

### 🌲 Model 1: LightGBM Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb

X_train, X_test, y_train, y_test = train_test_split(X_lexical, y, test_size=0.2, random_state=42)

print("🚀 Training LightGBM...")
train_data = lgb.Dataset(X_train, label=y_train)
params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1}
lgbm_model = lgb.train(params, train_data, num_boost_round=500)

y_pred_lgbm = lgbm_model.predict(X_test)
print(f"LGBM AUC-ROC: {roc_auc_score(y_test, y_pred_lgbm):.4f}")

### 🧠 Model 2: Neural Network Training

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn_model = tf.keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("🚀 Training Neural Network...")
nn_model.fit(X_train_scaled, y_train, validation_split=0.1, epochs=20, batch_size=64, verbose=1)

nn_loss, nn_acc = nn_model.evaluate(X_test_scaled, y_test)
print(f"NN Accuracy: {nn_acc:.4f}")

### 🤖 Model 3: DistilBERT Transformer Fine-tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_fn(examples):
    return tokenizer(examples['url'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(df_sample.iloc[:10000])
tokenized_ds = train_ds.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

print("🚀 Setup Trainer (Warnings can be ignored)... ")
# Minimal arguments to avoid version conflicts with 'evaluation_strategy' or 'eval_strategy'
training_args = TrainingArguments(
    output_dir='./results', 
    num_train_epochs=3, 
    per_device_train_batch_size=16,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds
)

print("🚀 Fine-tuning DistilBERT...")
trainer.train()
model.save_pretrained('./phishblocker_transformer')

### 💾 Save and Export

In [ ]:
import os
import json
from datetime import datetime

os.makedirs('models', exist_ok=True)

lgbm_model.save_model('models/phishblocker_lgb_model.txt')
nn_model.save('models/phishblocker_nn_model.h5')
joblib.dump(scaler, 'models/phishblocker_scaler.pkl')

metadata = {
    'feature_names': X_lexical.columns.tolist(),
    'ensemble_weights': {'lgb': 0.5, 'nn': 0.3, 'transformer': 0.2},
    'created_at': datetime.now().isoformat(),
    'accuracy': float(nn_acc)
}

with open('models/phishblocker_metadata.json', 'w') as f:
    json.dump(metadata, f)

!zip -r phishblocker_ensemble_v1.zip models phishblocker_transformer
print("✅ All models saved and zipped!")